# CNN vs ViT Visualization Notebook

This notebook is meant to make the project easy to inspect visually.

It lets you:
- optionally run the experiment pipeline from inside the notebook
- inspect the saved JSON summaries
- display the saved plots and interpretability images
- preview clean, occluded, and texture-modified CIFAR-10 samples


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"
INTERPRETABILITY_DIR = OUTPUT_DIR / "interpretability"

print(f"Project root: {PROJECT_ROOT}")
print(f"Outputs directory: {OUTPUT_DIR}")


## Optional: Run the Experiment Pipeline

Set `RUN_EXPERIMENTS = True` if you want the notebook to launch the project for you.

If you already ran `uv run python main.py`, leave it as `False` and go straight to visualization.


In [ ]:
RUN_EXPERIMENTS = False
RUN_ARGS = []

if RUN_EXPERIMENTS:
    command = ["uv", "run", "python", "main.py", *RUN_ARGS]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("Skipping experiment execution. Set RUN_EXPERIMENTS = True to launch it from the notebook.")


## Load Saved Summary

In [ ]:
def require(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing artifact: {path}. Run `uv run python main.py` first or set RUN_EXPERIMENTS = True above."
        )
    return path


def markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, separator, *body])


summary_path = require(OUTPUT_DIR / "summary.json")
with summary_path.open("r", encoding="utf-8") as handle:
    summary = json.load(handle)

print(f"Loaded summary from: {summary_path}")


In [ ]:
display(Markdown("## Baseline Results"))
display(
    Markdown(
        markdown_table(
            summary["baseline"],
            ["model", "train_fraction", "test_accuracy", "best_val_accuracy", "parameter_count", "training_time_readable"],
        )
    )
)

display(Markdown("## Robustness Results"))
display(
    Markdown(
        markdown_table(
            summary["robustness"],
            ["model", "shift", "clean_accuracy", "shifted_accuracy", "robustness_drop"],
        )
    )
)

display(Markdown("## Data Efficiency Results"))
display(
    Markdown(
        markdown_table(
            summary["data_efficiency"],
            ["model", "train_fraction", "train_size", "test_accuracy", "best_val_accuracy", "training_time_readable"],
        )
    )
)


## Image Display Helpers

In [ ]:
def show_images(image_paths, titles=None, cols=2, figsize=(14, 8)):
    image_paths = [Path(path) for path in image_paths]
    for path in image_paths:
        require(path)

    rows = (len(image_paths) + cols - 1) // cols
    figure, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for index, path in enumerate(image_paths):
        image = mpimg.imread(path)
        axes[index].imshow(image)
        title = titles[index] if titles else path.name
        axes[index].set_title(title)
        axes[index].axis("off")

    for index in range(len(image_paths), len(axes)):
        axes[index].axis("off")

    plt.tight_layout()
    plt.show()


## Training and Evaluation Plots

In [ ]:
show_images(
    [
        PLOTS_DIR / "cnn_training_curves.png",
        PLOTS_DIR / "vit_training_curves.png",
        PLOTS_DIR / "data_efficiency.png",
        PLOTS_DIR / "robustness_drop.png",
    ],
    titles=[
        "CNN training curves",
        "ViT training curves",
        "Data efficiency",
        "Robustness drop",
    ],
    cols=2,
    figsize=(16, 10),
)


## Interpretability Images

In [ ]:
show_images(
    [
        INTERPRETABILITY_DIR / "cnn_gradcam.png",
        INTERPRETABILITY_DIR / "vit_attention.png",
    ],
    titles=["CNN Grad-CAM", "ViT Attention Rollout"],
    cols=2,
    figsize=(16, 6),
)


## Preview Clean vs Occluded vs Texture-Modified CIFAR-10 Samples

This section rebuilds the dataset transforms so you can visually inspect what the robustness test data looks like.


In [ ]:
from configs.config import build_config
from datasets.cifar_loader import build_cifar_datasets
from datasets.occlusion import OcclusionWrapperDataset
from datasets.texture_modification import TextureModifiedDataset
from utils.helpers import to_numpy_image

config = build_config()

_, _, clean_test_dataset, class_names = build_cifar_datasets(
    config=config,
    train_fraction=1.0,
    test_variant="clean",
)

occluded_test_dataset = OcclusionWrapperDataset(
    clean_test_dataset,
    mask_size=config.augmentations.occlusion_mask_size,
    fill_value=config.augmentations.occlusion_fill_value,
    seed=config.training.seed,
)

texture_test_dataset = TextureModifiedDataset(
    clean_test_dataset,
    patch_size=config.augmentations.texture_patch_size,
    shuffle_fraction=config.augmentations.texture_shuffle_fraction,
    noise_std=config.augmentations.texture_noise_std,
    seed=config.training.seed,
)

print(f"Loaded {len(clean_test_dataset)} clean CIFAR-10 test images.")


In [ ]:
sample_index = 0

clean_image, label = clean_test_dataset[sample_index]
occluded_image, _ = occluded_test_dataset[sample_index]
texture_image, _ = texture_test_dataset[sample_index]

figure, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(to_numpy_image(clean_image, config.data.mean, config.data.std))
axes[0].set_title("Clean")
axes[0].axis("off")

axes[1].imshow(to_numpy_image(occluded_image, config.data.mean, config.data.std))
axes[1].set_title("Occluded")
axes[1].axis("off")

axes[2].imshow(to_numpy_image(texture_image, config.data.mean, config.data.std))
axes[2].set_title("Texture modified")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print(f"Sample index: {sample_index}")
print(f"Class label: {class_names[label]}")
